In [ ]:
!pip install transformers datasets accelerate gradio


In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

In [ ]:
import os
os.listdir()

In [ ]:
import pandas as pd

df = pd.read_csv("college_dataset.csv")   # change name if different

df = df.dropna()   # remove empty rows
df['text'] = df['text'].astype(str)  # convert everything to string
df.head()

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

num_labels = len(df['label'].unique())
print("Number of intents:", num_labels)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length')

dataset = dataset.map(tokenize, batched=True)

dataset = dataset.train_test_split(test_size=0.2)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    logging_dir="./logs",
    logging_steps=10
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

trainer.train()

In [ ]:
import torch.nn.functional as F

def predict_intent(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    confidence, pred = torch.max(probs, dim=1)

    print("Confidence:", confidence.item())  # DEBUG

    if confidence.item() < 0.2:
        return "fallback"

    return le.inverse_transform([pred.item()])[0]

In [ ]:
import random

responses = {
    "greeting": [
        "Hey bro 😄 what help do you need?",
        "Hello! How can I help you?",
        "Hey buddy, how are you?",
        "Tell me, what do you need help with? 🙌",
        "Hey! How can I help you, buddy?"
    ],

    "placement_drive": [
        "Placement updates are usually shared in the college placement group.",
        "You can get the latest info from the placement cell 👍",
        "Placements usually start from the third year 👍",
        "Try contacting placement cell members!",
        "The highest package has gone up to 50 LPA 😉",
        "The placement cell is on the 5th floor of the architecture building.",
        "Contact the placement cell for more information 👌"
    ],

    "college_fees": [
        "Fee details are available on the college website 💰",
        "The fee structure varies depending on your branch.",
        "Fees can be paid via cash or online. For more info, contact the admin office.",
        "Fees are usually paid at the start of the semester.",
        "You can pay fees online as well. Check with the admin office ✌️",
        "You can get the fee receipt from the admin building 😊",
        "All fee details are available on the college website 🙌"
    ],

    "exam_form": [
        "Exam forms are filled through the college portal 📅",
        "Make sure not to miss the last date 😅",
        "You will receive a mail when form filling starts 📩",
        "You will get a mail for backlog forms 📩",
        "You can check the last date on the portal 😉",
        "The exam form fee is around 3500 INR 🥲"
    ],

    "result_date": [
        "Results are usually declared 15–20 days after exams 📊",
        "You can check your result on the college portal.",
        "Marks are usually available within 15–20 days.",
        "Open day usually starts the next day after exams 😊"
    ],

    "attendance_criteria": [
        "A minimum of 75% attendance is required 📘",
        "Low attendance may cause issues in exam eligibility 😬",
        "Yes, 75% attendance is compulsory, otherwise you may not be allowed to sit for exams 🥲",
        "If your attendance is low, you may need to complete tasks given by your teacher 🥲",
        "Attendance is calculated based on your lecture attendance 😊"
    ],

    "campus_events": [
        "College fest updates are shared on the notice board or Instagram 🎉",
        "Events happen regularly, check the college page.",
        "Swartarang is usually held in February 🤩",
        "Event registration forms are shared in student groups 🙌",
        "College days usually start in the first week of February 🤩",
        "Innoveda is an IT department event where research papers are presented 🎊"
    ],

    "syllabus": [
        "The syllabus is available on the college website 📚",
        "You can download the official syllabus PDF online 📝",
        "You can find the syllabus on the PCCOE website 📝",
        "The syllabus is available on the college website 📋"
    ],

    "hostel_facilities": [
        "Hostel facilities are decent, and WiFi is available 🏠",
        "Usually, 3 people share one room ✌️",
        "You can find rules in the hostel rule book 📝",
        "Hostel and mess are both compulsory 😉",
        "Facilities like WiFi, laundry, AC/non-AC rooms are available (check website for details) 📝",
        "Hostel allotment is done after admission."
    ],

    "mess": [
        "Mess food is sometimes good, sometimes average 😂",
        "You can also try nearby canteens for better options 🍔",
        "Mess provides breakfast, lunch, evening snacks, and dinner 🤤",
        "Veg food is available in both mess facilities.",
        "Yes, mess is compulsory if you stay in the hostel 👌",
        "The canteen is located on the ground floor of the girls hostel and in the architecture basement 🤤",
        "Non-veg food is available in boys hostel and new girls hostel 😉",
        "Mess charges are included in the fee structure on the college website."
    ],

    "fallback": [
        "I didn’t understand that 😅 please rephrase.",
        "That seems out of scope 🤔",
        "I don’t have exact information on that 😬"
    ]
}

last_intent = None


In [ ]:
def chatbot_response(user_input):
    intent = predict_intent(user_input)
    return random.choice(responses.get(intent, responses["fallback"]))

In [ ]:
import gradio as gr

def chat(user_input):
    return chatbot_response(user_input)

gr.Interface(fn=chat, inputs="text", outputs="text").launch()